In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

### Reading the Table

In [0]:
# Read Silver tables
df_prd = spark.table("dev_project.silver.crm_prd_info") \
              

df_cat = spark.table("dev_project.silver.erp_px_cat_g1v2")

# Join on cat_id = category_id
df_dim_products = df_prd \
    .join(df_cat, df_prd["cat_id"] == df_cat["category_id"], how="left") \
    .select(
    "product_surrogate_key",
        "product_id",
        "product_key",
        "prd_key_short",
        "product_name",
        "cat_id",
        "category",
        "subcategory",
        "maintenance",
        "product_line",
        "product_cost",
        "start_date",
        "end_date", 
        "is_current"
    )

### Validation

In [0]:
print("\n" + "="*50)
print(" DIM_PRODUCTS VALIDATION")
print("="*50)
print(f"Total records:        {df_dim_products.count():,}")
print(f"Null surrogate keys:  {df_dim_products.filter(F.col('product_surrogate_key').isNull()).count()}")
print(f"Null product keys:    {df_dim_products.filter(F.col('product_key').isNull()).count()}")
print(f"Null categories:      {df_dim_products.filter(F.col('category').isNull()).count()}")
print(f"Category breakdown:")
df_dim_products.groupBy("category").count().show()
print(f"Product line breakdown:")
df_dim_products.groupBy("product_line").count().show()
print("="*50)

In [0]:
df_dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dev_project.gold.dim_products")

print("dim_products Gold write complete.")